# Sistema de Matching de Planillas PILA con Log Financiero

Este notebook procesa todas las planillas en el directorio /Planillas y las compara con el log financiero de Excel.

## Criterios de Matching:
- NIT del aportante (coincidencia exacta)
- Valor monetario (tolerancia ±5%)

## Parsers utilizados:
- tipo1: Encabezados
- tipo2: Detalles
- tipo3: Renglones 31, 36, 39 (MOSTRANDO TODOS LOS CAMPOS DISPONIBLES)
- tipo4: Datos adicionales

In [ ]:
import pandas as pd
import os
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.parsers.txt_parser import parsear_tipo1, parsear_tipo2, parsear_tipo3, parsear_tipo4

## 1. Función para procesar todas las planillas

In [ ]:
def procesar_todas_planillas(directorio):
    """
    Procesa todos los archivos .txt de planillas usando todos los parsers disponibles.
    
    Args:
        directorio: Ruta al directorio con los archivos .txt
        
    Returns:
        Dict con DataFrames de cada tipo de parser
    """
    resultados = {
        'encabezados': [],       # parsear_tipo1
        'detalles': [],          # parsear_tipo2  
        'totales_r31': [],       # parsear_tipo3 - renglon_31_aportes
        'totales_r36': [],       # parsear_tipo3 - renglon_36_mora
        'totales_r39': [],       # parsear_tipo3 - renglon_39_total
        'datos_adicionales': [], # parsear_tipo4
        'errores': []
    }
    
    # Obtener todos los archivos .txt
    archivos_txt = [f for f in os.listdir(directorio) if f.endswith('.txt')]
    
    print(f"📁 Encontrados {len(archivos_txt)} archivos .txt\n")
    
    for archivo in archivos_txt:
        ruta_completa = os.path.join(directorio, archivo)
        print(f"\n📄 Procesando: {archivo}")
        
        try:
            # Intentar leer con diferentes encodings
            contenido = None
            for encoding in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
                try:
                    with open(ruta_completa, 'r', encoding=encoding) as f:
                        contenido = f.read()
                    print(f"  ✓ Leído con encoding: {encoding}")
                    break
                except UnicodeDecodeError:
                    continue
            
            if contenido is None:
                resultados['errores'].append({
                    'archivo': archivo,
                    'error': 'No se pudo leer con ningún encoding'
                })
                continue
            
            # Parser tipo1 - Encabezados
            try:
                df_enc = parsear_tipo1(contenido)
                df_enc['archivo_origen'] = archivo
                resultados['encabezados'].append(df_enc)
                print(f"  ✓ Tipo1 (Encabezados): {len(df_enc)} registro(s)")
            except Exception as e:
                print(f"  ✗ Error en tipo1: {e}")
            
            # Parser tipo2 - Detalles
            try:
                df_det = parsear_tipo2(contenido)
                df_det['archivo_origen'] = archivo
                resultados['detalles'].append(df_det)
                print(f"  ✓ Tipo2 (Detalles): {len(df_det)} registro(s)")
            except Exception as e:
                print(f"  ✗ Error en tipo2: {e}")
            
            # Parser tipo3 - Renglones (31, 36, 39)
            try:
                df_r31, df_r36, df_r39 = parsear_tipo3(contenido)
                df_r31['archivo_origen'] = archivo
                df_r36['archivo_origen'] = archivo
                df_r39['archivo_origen'] = archivo
                resultados['totales_r31'].append(df_r31)
                resultados['totales_r36'].append(df_r36)
                resultados['totales_r39'].append(df_r39)
                print(f"  ✓ Tipo3 (Renglones): R31, R36, R39 procesados")
            except Exception as e:
                print(f"  ✗ Error en tipo3: {e}")
            
            # Parser tipo4 - Datos adicionales
            try:
                df_adic = parsear_tipo4(contenido)
                df_adic['archivo_origen'] = archivo
                resultados['datos_adicionales'].append(df_adic)
                print(f"  ✓ Tipo4 (Adicionales): {len(df_adic)} registro(s)")
            except Exception as e:
                print(f"  ✗ Error en tipo4: {e}")
                
        except Exception as e:
            resultados['errores'].append({
                'archivo': archivo,
                'error': str(e)
            })
            print(f"  ✗ Error general: {e}")
    
    # Consolidar DataFrames
    print("\n" + "="*50)
    print("📊 CONSOLIDANDO RESULTADOS...")
    print("="*50)
    
    df_encabezados = pd.concat(resultados['encabezados'], ignore_index=True) if resultados['encabezados'] else pd.DataFrame()
    df_detalles = pd.concat(resultados['detalles'], ignore_index=True) if resultados['detalles'] else pd.DataFrame()
    df_r31 = pd.concat(resultados['totales_r31'], ignore_index=True) if resultados['totales_r31'] else pd.DataFrame()
    df_r36 = pd.concat(resultados['totales_r36'], ignore_index=True) if resultados['totales_r36'] else pd.DataFrame()
    df_r39 = pd.concat(resultados['totales_r39'], ignore_index=True) if resultados['totales_r39'] else pd.DataFrame()
    df_adicionales = pd.concat(resultados['datos_adicionales'], ignore_index=True) if resultados['datos_adicionales'] else pd.DataFrame()
    
    print(f"✓ Encabezados: {len(df_encabezados)} registros")
    print(f"✓ Detalles: {len(df_detalles)} registros")
    print(f"✓ Renglon 31: {len(df_r31)} registros")
    print(f"✓ Renglon 36: {len(df_r36)} registros")
    print(f"✓ Renglon 39: {len(df_r39)} registros")
    print(f"✓ Datos adicionales: {len(df_adicionales)} registros")
    print(f"✗ Errores: {len(resultados['errores'])} archivo(s)\n")
    
    return {
        'df_encabezados': df_encabezados,
        'df_detalles': df_detalles,
        'df_r31': df_r31,
        'df_r36': df_r36,
        'df_r39': df_r39,
        'df_adicionales': df_adicionales,
        'errores': resultados['errores']
    }

## 2. Función para analizar el log financiero de Excel

In [ ]:
def analizar_log_financiero(ruta_excel):
    """
    Analiza el archivo Excel de log financiero.
    
    Args:
        ruta_excel: Ruta al archivo Excel
        
    Returns:
        Dict con información del log
    """
    print("\n" + "="*50)
    print("📊 ANALIZANDO LOG FINANCIERO")
    print("="*50)
    
    # Leer todas las hojas
    xls = pd.ExcelFile(ruta_excel)
    print(f"\nHojas encontradas: {xls.sheet_names}\n")
    
    logs_data = {}
    
    for sheet_name in xls.sheet_names:
        df = pd.read_excel(ruta_excel, sheet_name=sheet_name)
        logs_data[sheet_name] = df
        
        print(f"📄 Hoja: {sheet_name}")
        print(f"   Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
        print(f"   Columnas: {list(df.columns[:5])}{'...' if len(df.columns) > 5 else ''}\n")
    
    return logs_data

## 3. Función de matching entre planillas y log

### Criterios de matching (deben cumplirse AMBOS):
1. **NIT exacto**: El NIT del aportante debe aparecer en alguna columna de texto del log
2. **Valor similar**: El valor total debe coincidir con ±5% de tolerancia en alguna columna numérica

In [ ]:
def buscar_matches_planillas_log(df_encabezados, df_detalles, df_r39, logs_data):
    """
    Busca coincidencias entre planillas y log financiero.
    Requiere que se cumplan AMBOS criterios simultáneamente:
    1. Coincidencia exacta de NIT
    2. Coincidencia de valor monetario (±5%)
    
    Args:
        df_encabezados: DataFrame con encabezados
        df_detalles: DataFrame con detalles
        df_r39: DataFrame con totales (Renglon 39)
        logs_data: Dict con DataFrames del log por hoja
        
    Returns:
        DataFrame con resultados de matching
    """
    print("\n" + "="*50)
    print("🔍 BUSCANDO MATCHES (2 CRITERIOS: NIT + VALOR)")
    print("="*50)
    
    resultados_matching = []
    
    # Combinar encabezados con totales R39
    df_planillas = pd.merge(
        df_encabezados,
        df_r39[['archivo_origen', 'valor_total', 'neto_pagar']],
        on='archivo_origen',
        how='left'
    )
    
    print(f"\nTotal planillas a procesar: {len(df_planillas)}\n")
    
    for idx, planilla in df_planillas.iterrows():
        archivo = planilla['archivo_origen']
        nit_aportante = str(planilla.get('nit_aportante', '')).strip()
        valor_total = planilla.get('valor_total', 0) or planilla.get('neto_pagar', 0)
        
        print(f"\n{'='*50}")
        print(f"Planilla: {archivo}")
        print(f"NIT: {nit_aportante}")
        print(f"Valor: ${valor_total:,.2f}")
        
        # Validar datos completos
        if not nit_aportante or valor_total <= 0:
            resultado = {
                'archivo': archivo,
                'nit': nit_aportante,
                'valor_planilla': valor_total,
                'match_encontrado': False,
                'razon_no_match': 'Datos incompletos (NIT o valor faltante)',
                'hoja_log': None,
                'fila_log': None,
                'valor_log': None
            }
            resultados_matching.append(resultado)
            print("❌ Datos incompletos")
            continue
        
        # Buscar en cada hoja del log
        match_encontrado = False
        tolerance = valor_total * 0.05  # 5% de tolerancia
        
        for sheet_name, df_log in logs_data.items():
            if df_log.empty:
                continue
            
            print(f"\n  🔎 Buscando en hoja: {sheet_name}")
            
            # PASO 1: Buscar coincidencias de NIT en todas las columnas de texto
            nit_matches = set()
            for col in df_log.columns:
                if df_log[col].dtype == 'object':
                    matches_nit = df_log[df_log[col].astype(str).str.contains(nit_aportante, na=False, case=False)]
                    if not matches_nit.empty:
                        nit_matches.update(matches_nit.index.tolist())
                        print(f"    ✓ NIT encontrado en columna '{col}': {len(matches_nit)} coincidencia(s)")
            
            if not nit_matches:
                print(f"    ✗ NIT no encontrado en esta hoja")
                continue
            
            print(f"    → Total filas con NIT: {len(nit_matches)}")
            
            # PASO 2: Buscar coincidencias de VALOR solo en las filas que tienen el NIT
            df_log_nit = df_log.loc[list(nit_matches)]
            valor_matches = set()
            
            for col in df_log_nit.columns:
                if pd.api.types.is_numeric_dtype(df_log_nit[col]):
                    matches_valor = df_log_nit[
                        (df_log_nit[col] >= valor_total - tolerance) & 
                        (df_log_nit[col] <= valor_total + tolerance)
                    ]
                    if not matches_valor.empty:
                        valor_matches.update(matches_valor.index.tolist())
                        print(f"    ✓ Valor encontrado en columna '{col}': {len(matches_valor)} coincidencia(s)")
            
            # PASO 3: Intersección - Solo filas que cumplen AMBOS criterios
            matches_completos = nit_matches.intersection(valor_matches)
            
            if matches_completos:
                print(f"\n    ✅ MATCH COMPLETO: {len(matches_completos)} fila(s) cumplen NIT + VALOR")
                
                for fila_idx in matches_completos:
                    fila = df_log.loc[fila_idx]
                    
                    # Encontrar el valor exacto que hizo match
                    valor_log = None
                    for col in df_log.columns:
                        if pd.api.types.is_numeric_dtype(df_log[col]):
                            val = fila[col]
                            if pd.notna(val) and abs(val - valor_total) <= tolerance:
                                valor_log = val
                                break
                    
                    resultado = {
                        'archivo': archivo,
                        'nit': nit_aportante,
                        'valor_planilla': valor_total,
                        'match_encontrado': True,
                        'razon_no_match': None,
                        'hoja_log': sheet_name,
                        'fila_log': fila_idx,
                        'valor_log': valor_log,
                        'diferencia_valor': abs(valor_total - valor_log) if valor_log else None,
                        'porcentaje_diferencia': abs((valor_total - valor_log) / valor_total * 100) if valor_log and valor_total > 0 else None
                    }
                    resultados_matching.append(resultado)
                    match_encontrado = True
                    print(f"      Fila {fila_idx}: Valor log ${valor_log:,.2f}")
            else:
                print(f"    ✗ No hay filas que cumplan AMBOS criterios simultáneamente")
                if nit_matches and not valor_matches:
                    print(f"      (Se encontró NIT pero no el valor)")
        
        # Si no se encontró match en ninguna hoja
        if not match_encontrado:
            resultado = {
                'archivo': archivo,
                'nit': nit_aportante,
                'valor_planilla': valor_total,
                'match_encontrado': False,
                'razon_no_match': 'No se cumplieron AMBOS criterios (NIT + valor) en ninguna fila',
                'hoja_log': None,
                'fila_log': None,
                'valor_log': None
            }
            resultados_matching.append(resultado)
            print("\n❌ NO MATCH - Criterios no cumplidos simultáneamente")
    
    return pd.DataFrame(resultados_matching)

## 4. EJECUCIÓN PRINCIPAL

In [ ]:
# Configuración de rutas
DIRECTORIO_PLANILLAS = "../Planillas"
RUTA_LOG_EXCEL = "../Planillas/LOG FINANCIERO FPOB OCTUBRE .xlsx"

# Verificar que existan los directorios
if not os.path.exists(DIRECTORIO_PLANILLAS):
    print(f"⚠️  Directorio no encontrado: {DIRECTORIO_PLANILLAS}")
    print("   Creando directorio...")
    os.makedirs(DIRECTORIO_PLANILLAS, exist_ok=True)

if not os.path.exists(RUTA_LOG_EXCEL):
    print(f"⚠️  Archivo Excel no encontrado: {RUTA_LOG_EXCEL}")
    print("   Por favor, coloca el archivo en el directorio /Planillas")

In [ ]:
# Paso 1: Procesar todas las planillas
resultados = procesar_todas_planillas(DIRECTORIO_PLANILLAS)

df_encabezados = resultados['df_encabezados']
df_detalles = resultados['df_detalles']
df_r31 = resultados['df_r31']
df_r36 = resultados['df_r36']
df_r39 = resultados['df_r39']
df_adicionales = resultados['df_adicionales']

## 5. VISUALIZACIÓN DE DATOS DEL PARSER TIPO3

### ⭐ MOSTRANDO TODOS LOS CAMPOS DISPONIBLES ⭐

Esta sección muestra TODOS los campos parseados por el parser tipo3 para cada DataFrame (R31, R36, R39).

In [ ]:
print("\n" + "="*70)
print("📊 RENGLON 31 - APORTES (TODOS LOS CAMPOS)")
print("="*70)

if not df_r31.empty:
    # Configurar pandas para mostrar todas las columnas
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)
    
    print(f"\nTotal de registros: {len(df_r31)}")
    print(f"\nCampos disponibles ({len(df_r31.columns)}):")
    for i, col in enumerate(df_r31.columns, 1):
        print(f"  {i}. {col}")
    
    print("\n" + "-"*70)
    print("DATOS COMPLETOS:")
    print("-"*70)
    display(df_r31)
    
    print("\n" + "-"*70)
    print("INFORMACIÓN DEL DATAFRAME:")
    print("-"*70)
    df_r31.info()
    
    print("\n" + "-"*70)
    print("ESTADÍSTICAS DESCRIPTIVAS:")
    print("-"*70)
    display(df_r31.describe())
else:
    print("\n⚠️  No hay datos en Renglon 31")

In [ ]:
print("\n" + "="*70)
print("📊 RENGLON 36 - INTERESES DE MORA (TODOS LOS CAMPOS)")
print("="*70)

if not df_r36.empty:
    print(f"\nTotal de registros: {len(df_r36)}")
    print(f"\nCampos disponibles ({len(df_r36.columns)}):")
    for i, col in enumerate(df_r36.columns, 1):
        print(f"  {i}. {col}")
    
    print("\n" + "-"*70)
    print("DATOS COMPLETOS:")
    print("-"*70)
    display(df_r36)
    
    print("\n" + "-"*70)
    print("INFORMACIÓN DEL DATAFRAME:")
    print("-"*70)
    df_r36.info()
    
    print("\n" + "-"*70)
    print("ESTADÍSTICAS DESCRIPTIVAS:")
    print("-"*70)
    display(df_r36.describe())
else:
    print("\n⚠️  No hay datos en Renglon 36")

In [ ]:
print("\n" + "="*70)
print("📊 RENGLON 39 - TOTAL A PAGAR (TODOS LOS CAMPOS)")
print("="*70)

if not df_r39.empty:
    print(f"\nTotal de registros: {len(df_r39)}")
    print(f"\nCampos disponibles ({len(df_r39.columns)}):")
    for i, col in enumerate(df_r39.columns, 1):
        print(f"  {i}. {col}")
    
    print("\n" + "-"*70)
    print("DATOS COMPLETOS:")
    print("-"*70)
    display(df_r39)
    
    print("\n" + "-"*70)
    print("INFORMACIÓN DEL DATAFRAME:")
    print("-"*70)
    df_r39.info()
    
    print("\n" + "-"*70)
    print("ESTADÍSTICAS DESCRIPTIVAS:")
    print("-"*70)
    display(df_r39.describe())
else:
    print("\n⚠️  No hay datos en Renglon 39")

## 6. Análisis del Log Financiero

In [ ]:
# Paso 2: Analizar log financiero
if os.path.exists(RUTA_LOG_EXCEL):
    logs_data = analizar_log_financiero(RUTA_LOG_EXCEL)
else:
    print("⚠️  Archivo Excel no encontrado. Saltando análisis del log.")
    logs_data = {}

## 7. Matching de Planillas con Log

In [ ]:
# Paso 3: Ejecutar matching
if logs_data and not df_encabezados.empty and not df_r39.empty:
    df_matches = buscar_matches_planillas_log(df_encabezados, df_detalles, df_r39, logs_data)
else:
    print("⚠️  No se puede ejecutar matching - faltan datos")
    df_matches = pd.DataFrame()

## 8. Resumen de Resultados

In [ ]:
if not df_matches.empty:
    print("\n" + "="*70)
    print("📈 ESTADÍSTICAS DE MATCHING")
    print("="*70)
    
    total_planillas = len(df_matches['archivo'].unique())
    matches_exitosos = df_matches[df_matches['match_encontrado'] == True]
    matches_fallidos = df_matches[df_matches['match_encontrado'] == False]
    
    print(f"\nTotal planillas procesadas: {total_planillas}")
    print(f"Matches exitosos: {len(matches_exitosos)} ({len(matches_exitosos)/len(df_matches)*100:.1f}%)")
    print(f"Sin match: {len(matches_fallidos)} ({len(matches_fallidos)/len(df_matches)*100:.1f}%)")
    
    if not matches_exitosos.empty:
        print("\n" + "-"*70)
        print("PLANILLAS CON MATCH:")
        print("-"*70)
        display(matches_exitosos[['archivo', 'nit', 'valor_planilla', 'hoja_log', 'valor_log', 'porcentaje_diferencia']])
    
    if not matches_fallidos.empty:
        print("\n" + "-"*70)
        print("PLANILLAS SIN MATCH:")
        print("-"*70)
        display(matches_fallidos[['archivo', 'nit', 'valor_planilla', 'razon_no_match']])
    
    # Análisis de razones de no-match
    if not matches_fallidos.empty:
        print("\n" + "-"*70)
        print("DISTRIBUCIÓN DE RAZONES DE NO-MATCH:")
        print("-"*70)
        razones = matches_fallidos['razon_no_match'].value_counts()
        for razon, count in razones.items():
            print(f"  • {razon}: {count} caso(s)")
else:
    print("\n⚠️  No hay resultados de matching para mostrar")

## 9. Exportar Resultados (Opcional)

In [ ]:
# Exportar resultados a Excel
if not df_matches.empty:
    output_file = "../Planillas/resultados_matching.xlsx"
    
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df_matches.to_excel(writer, sheet_name='Matches', index=False)
        df_r31.to_excel(writer, sheet_name='Renglon_31_Aportes', index=False)
        df_r36.to_excel(writer, sheet_name='Renglon_36_Mora', index=False)
        df_r39.to_excel(writer, sheet_name='Renglon_39_Total', index=False)
        df_encabezados.to_excel(writer, sheet_name='Encabezados', index=False)
    
    print(f"\n✅ Resultados exportados a: {output_file}")
else:
    print("\n⚠️  No hay resultados para exportar")